In [1]:
using Pkg
Pkg.activate("./")
Pkg.develop(path="../../")
ENV["TAMBOSIM_PATH"] = realpath("../../")

  Activating project at `~/research/TAMBO-MC/notebooks/create_geometry`
   Resolving package versions...
     Project No packages added to or removed from `~/research/TAMBO-MC/notebooks/create_geometry/Project.toml`
    Manifest No packages added to or removed from `~/research/TAMBO-MC/notebooks/create_geometry/Manifest.toml`


"/Users/jlazar/research/TAMBO-MC"

In [ ]:
using Tambo
using Makie
using CairoMakie
using HDF5
using LinearAlgebra
using Unitful
using StatsBase
include("../plotting_boilerplate.jl")

In [ ]:
filename = "./triangulation.h5"
key = "colca_valley_30000"

earth = Tambo.Earth("$(filename):$(key)")
enu_coordinates = Tambo.CoordinateSystem(earth)

In [ ]:
outline = Tuple{Float64, Float64}[]
for line in readlines("./TAMBO_outline.csv")
    long, lat = deg2rad.(parse.(Float64, split(line, ", ")))
    push!(outline, (long, lat))
end
    
# This need to be oriented anti-clockwise
cog = [sum([x[1] for x in outline]), sum([x[2] for x in outline])] ./ length(outline)
phis = []
for (x,y) in outline
    push!(phis, atan(y-cog[2], x-cog[1]))
end
sorter = sortperm(phis)
outline = outline[sorter]

In [ ]:
coords = Tambo.Coordinate[]
for longlat in outline
    x, y, z = Tambo.longlat_to_cart(longlat...) .* earth.prem[end].radius
    c = Tambo.Coordinate(x, y, z, Tambo.ecefcoordinates)
    c = convert(enu_coordinates, c)
    push!(coords, c)
end

In [ ]:
radius = 6 * u"km"
fig = Figure()

triangles = filter(triangle->norm(Tambo.centroid(triangle)[1:2]) < radius, earth.topography)
triangles = filter(triangle->Tambo.centroid(triangle)[3] > 0 * u"km", triangles)

vxs, faces = Tambo.triangles_to_mesh(triangles)

zaspect = 4500*Tambo.uparse("m") / (2 * radius)

ax = Axis3(
    fig[1, 1],
    azimuth=deg2rad(-90),
    elevation=deg2rad(90),
    aspect=(1, 1, zaspect),
    zgridvisible=false,
    zlabelvisible=false,
    zticklabelsvisible=false,
    zticksvisible=false,
    xlabel="x [m]",
    ylabel="y [m]",
)
zlims!(ax, 1000, 5500)

m = mesh!(
    ax,
    map(vx->Point3f(ustrip.(vx.point)), vxs),
    faces,
    color=[v.point.z.val for v in vxs],
    colormap=Reverse(:speed),
    colorrange=[2000, 4000]
)

lines!(
    ax,
    ustrip.(vcat([c[1] for c in coords], [coords[1][1]])),
    ustrip.(vcat([c[2] for c in coords], [coords[1][2]])),
    [2000.0 for _ in 1:length(outline)+1],
    linewidth=5,
    color=:crimson
)

# Axis 3: colorbar
colorbar = Colorbar(fig[1,2], m, label="Elevation [m]")

display(fig)

In [ ]:
function is_in_outline(
    outline_points::Vector{Tuple{Float64,Float64}}, 
    test_point::Tuple{Float64,Float64}
)
    
    n = length(outline_points)
    inside = false

    j = n
    for i in 1:n
        xi, yi = outline_points[i]
        xj, yj = outline_points[j]
        px, py = test_point

        # Check if point is between the y-coordinates of the edge
        if ((yi > py) != (yj > py)) &&
           (px < (xj - xi) * (py - yi) / (yj - yi) + xi)
#                 inside = true
            inside = !inside
        end
        j = i
    end
    
    return inside
end

In [ ]:
zaspect = 4500*Tambo.uparse("m") / (2 * radius)
    
fig = Figure()
ax = Axis3(
    fig[1, 1],
    azimuth=deg2rad(-80),
    elevation=deg2rad(20),
    aspect=(1, 1, zaspect),
    zgridvisible=false,
    zlabelvisible=false,
    zticklabelsvisible=false,
    zticksvisible=false,
    xlabel="x [m]",
    ylabel="y [m]",
#         margins = (50, 50, 50, 50)
)
zlims!(ax, 1000, 5500)
    
triangles = filter(triangle->norm(Tambo.centroid(triangle)[1:2]) < radius, earth.topography)
triangles = filter(triangle->Tambo.centroid(triangle)[3] > 0 * u"km", triangles)

vxs, faces = Tambo.triangles_to_mesh(triangles)

m = mesh!(
    ax,
    map(vx->Point3f(ustrip.(vx.point)), vxs),
    faces,
    color=[v.point.z.val for v in vxs],
    colormap=Reverse(:speed),
    colorrange=[2000, 4000],
    alpha=0.1
)

longlats = Tuple{Float64, Float64}[]
for triangle in earth.topography
    c = Tambo.centroid(triangle)
    long, lat = Tambo.cart_to_longlat(c)
    push!(longlats, (long, lat))
end

triangles = earth.topography[is_in_outline.(Ref(outline), longlats)]

vxs, faces = Tambo.triangles_to_mesh(triangles)

m = mesh!(
    ax,
    map(vx->Point3f(ustrip.(vx.point)), vxs),
    faces,
    color=[v.point.z.val for v in vxs],
    colormap=Reverse(:speed),
    colorrange=[2000, 4000]
) 

# Axis 3: colorbar
colorbar = Colorbar(fig[1,2], m, label="Elevation [m]")
    
fig

In [ ]:
idxs = findall(is_in_outline.(Ref(outline), longlats))
h5open(filename, "r+") do file
    group = file[key]
    group["detector1"] = idxs
end